# BanglaSQL — Colab Training Notebook
**Natural Language (Bangla) to SQL — University Management System**

### Steps
Run each cell **in order**. Runtime → Change runtime type → **T4 GPU** before starting.

## Step 1 — Check GPU

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU')

## Step 2 — Clone repository

In [ ]:
# Replace with your actual GitHub repo URL
REPO_URL = 'https://github.com/YOUR_USERNAME/YOUR_REPO_NAME.git'

!git clone {REPO_URL} banglasql
%cd banglasql/
!ls -la

## Step 3 — Install dependencies

In [ ]:
!pip install -r requirements.txt -q
print('Dependencies installed.')

## Step 4 — Generate database & dataset

In [ ]:
!python create_database.py

In [ ]:
!python build_dataset.py

## Step 5 — Tokenizer analysis & preprocessing

In [ ]:
!python preprocess_check.py

## Step 6 — Train
> Expected time: ~30–60 min on T4 GPU for 20 epochs over ~433 training pairs.

In [ ]:
!python train.py

## Step 7 — Save model to Google Drive (prevents loss on session timeout)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

DRIVE_SAVE_DIR = '/content/drive/MyDrive/BanglaSQL/checkpoints'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

# Copy the best model checkpoint
src = 'checkpoints/best_model'
dst = os.path.join(DRIVE_SAVE_DIR, 'best_model')

if os.path.exists(src):
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f'Best model saved to Google Drive: {dst}')
else:
    print('No best_model found. Check that training completed successfully.')

## Step 8 — Quick inference test (verify the model works)

In [ ]:
import json, unicodedata
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load config
with open('data/train_config.json') as f:
    cfg = json.load(f)

MODEL_DIR    = 'checkpoints/best_model'
SCHEMA       = cfg['schema_string']
MAX_IN       = cfg['max_input_length']
MAX_OUT      = cfg['max_target_length']

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR)
model.eval()

def predict(bangla_question: str) -> str:
    q   = unicodedata.normalize('NFC', bangla_question.strip())
    inp = f'translate Bangla to SQL: {q} </s> {SCHEMA}'
    ids = tokenizer(inp, return_tensors='pt', max_length=MAX_IN, truncation=True)
    out = model.generate(**ids, max_length=MAX_OUT, num_beams=4, early_stopping=True)
    return tokenizer.decode(out[0], skip_special_tokens=True)

# Test questions
test_questions = [
    'সকল শিক্ষার্থীর তালিকা দাও।',
    'যেসব শিক্ষার্থীর CGPA ৩.৫-এর বেশি তাদের নাম দাও।',
    'প্রতিটি বিভাগে কতজন শিক্ষার্থী আছে তা দেখাও।',
]

print('=== Inference Test ===\n')
for q in test_questions:
    sql = predict(q)
    print(f'Q: {q}')
    print(f'SQL: {sql}')
    print()

## Step 9 — Download model (alternative to Drive)
If you prefer to download the checkpoint directly:

In [ ]:
import shutil
shutil.make_archive('best_model', 'zip', 'checkpoints/best_model')

from google.colab import files
files.download('best_model.zip')